In [1]:
# dependecies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
data = pd.read_csv('/content/drive/MyDrive/cleaned-data.csv')

data = data.drop(['Unnamed: 0'], axis=1)

data.head()

In [5]:
data.info()

In [6]:
data.describe()

In [7]:
data.isnull().sum()

In [8]:
data.duplicated().sum()

In [9]:
data.shape

In [10]:
import seaborn as sns
from matplotlib.colors import ListedColormap 

In [11]:
# for all my plots
palette = ['#EAD3A9', '#84592B', '#A05135', 
           '#743015', '#462D1B', '#9D9368']
customcmap = ListedColormap(palette)

In [12]:
# creating cross table to evaluate if admission is a valuable variable
crosstab01 = pd.crosstab(data['admission_flag'], 
                         data['patient_waittime'])

plt.figure(figsize=(12, 7))
crosstab01.plot(kind='bar', stacked=True, colormap=customcmap)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [13]:
# determined by the previous images, admission_flag 
# has inconclusive context and 
# thus need to be excluded from any further analysis or modeling.
data = data.drop(['admission_flag'], axis=1)
data.head()

In [14]:
histogram = data.select_dtypes(
    include=['number', 'bool']).hist(bins=15, figsize=[12, 6])
plt.show()

In [15]:
print(data['patient_race'].unique())
print(data['department_referral'].unique())

In [16]:
data['patient_race'].value_counts(normalize=True)

In [17]:
data['department_referral'].value_counts(normalize=True)

In [18]:
# this code allows me to only see the people who were referred to a department
referred = data.loc[data[
    'department_referral'] != 'No referral.', 'department_referral']       

In [19]:
referred.value_counts(normalize=True)

In [20]:
data['patient_race'].value_counts(
    normalize=True).mul(100).round(1)

In [ ]:
# is referred but adds race for proper percentage calculations
referrals = data.loc[
    data['department_referral'].ne('No referral.'),
    ['patient_race', 'department_referral']
]

# the percentage of patients of that race referred to that department
crosstab_race_dpt = (
    pd.crosstab(
        referrals['patient_race'],
        referrals['department_referral'],
        normalize='columns' # index means to sums to 100 for the row; columns for the column
    )
    .mul(100)
)
# for index: given a patient is of a certain race, what percentage of their referrals go to each department
# for columns: given a pateint was referred to a certain department, what's the racial breakdown of that department
crosstab_race_dpt.round(1)

In [36]:
referrals['patient_race'].value_counts(
    normalize=True).mul(100).round(1)

In [ ]:
baseline = (
    referrals['department_referral'].value_counts(
        normalize=True).
    mul(100).
    reindex(crosstab_race_dpt.columns)
)

difference_pp = crosstab_race_dpt.sub(
    baseline, axis='columns')
difference_pp.round(2)

In [23]:
print(baseline)

In [24]:
plot_data = difference_pp.T

fig, ax = plt.subplots(figsize=(13, 7))

for race in plot_data.columns:
    values = plot_data[race]
    colors = values.map(lambda x: '#2e8b57' if x >= 0 else '#c94c4c')
    ax.barh(plot_data.index, values, left=0, alpha=0.55, color=colors)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Difference from overall referral baseline (percentage points)')
ax.set_ylabel('Department')
plt.tight_layout()
plt.show()

In [25]:
department_total_difference = (
    difference_pp
    .abs()
    .sum(axis=0)
    .sort_values(ascending=False)
)

department_total_difference.round(1)

In [26]:
race_weights = referrals['patient_race'].value_counts(normalize=True)

department_weighted_difference = (
    difference_pp
    .abs()
    .mul(race_weights, axis=0)
    .sum(axis=0)
    .sort_values(ascending=False)
)

department_weighted_difference.round(1)

In [ ]:
# Keep full precision for calculations; round only displayed labels.
difference_pp = crosstab_race_dpt.sub(baseline, axis='columns')

races = difference_pp.index
departments = difference_pp.columns
x_limit = np.ceil(difference_pp.abs().to_numpy().max() + 0.5)

fig, axes = plt.subplots(4, 2, figsize=(16, 18), sharex=True)
axes = axes.flatten()

for ax, race in zip(axes, races):
    values = difference_pp.loc[race].sort_values()
    colors = np.where(values >= 0, '#3b8f5a', '#c95050')

    ax.barh(values.index, values, color=colors)
    ax.axvline(0, color='black', linewidth=1)

    for department, value in values.items():
        label = f'{value:+.2f} pp'
        x_position = value + (0.12 if value >= 0 else -0.12)
        alignment = 'left' if value >= 0 else 'right'

        ax.text(
            x_position, department, label,
            va='center', ha=alignment, fontsize=9
        )

    ax.set_title(race)
    ax.set_xlim(-x_limit, x_limit)
    ax.set_xlabel('Difference from department baseline (percentage points)')
    ax.set_ylabel('')

# Remove the unused eighth panel.
fig.delaxes(axes[-1])

fig.suptitle(
    'Referral Distribution by Race vs. Overall Referral Baseline',
    fontsize=16,
    y=1.01
)
plt.tight_layout()
plt.show()

In [28]:
plt.figure(figsize=(13, 6))

sns.heatmap(
    difference_pp,
    annot=difference_pp.round(2),
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Difference from baseline (percentage points)'}
)

plt.xlabel('Department referral')
plt.ylabel('Patient race')
plt.title('Race-Specific Referral Mix Compared with Overall Referral Baseline')
plt.tight_layout()
plt.show()

In [29]:
referrals_waittime = data.loc[
    data['department_referral'].ne('No referral.'),
    ['patient_waittime', 'department_referral']
]

In [30]:
mean_values = referrals_waittime.groupby('department_referral')['patient_waittime'].mean()

plt.figure(figsize=(12, 6))
mean_values.plot(kind='bar', color=customcmap.colors)
plt.title('Average Waittime per Department Referral')
plt.xlabel('Department Referral')
plt.ylabel('Patient Waittime')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [31]:
print(data.dtypes)

In [32]:
data.describe()

In [33]:
columns = ['patient_race', 'patient_gender', 'age_category', 'day_of_week', 'time_category']
for col in columns:
    mean_values = data.groupby(col)['patient_waittime'].mean()

    plt.figure(figsize=(12, 6))
    mean_values.plot(kind='bar', color=customcmap.colors)
    plt.title(f"Average waittime per {col.capitalize()}")
    plt.xlabel(col.capitalize())
    plt.ylabel('Patient Waittime')
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [34]:
mean_values = data.groupby('patient_race')['patient_age'].mean()

plt.figure(figsize=(12, 6))
mean_values.plot(kind='bar', color=customcmap.colors)
plt.title('Average Age per Patient Race')
plt.xlabel('Patient Race')
plt.ylabel('Average Age')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()